# Train-only split ablation: inference vs fine-tuned retrieval vs learned EdgeMLP

This notebook creates an **internal train/test split** using **only** `data/train/` and compares clustering quality across:

1) **Inference-only**: pretrained backbone embeddings + cosine edge weights (no trained checkpoints)
2) **Fine-tuned DINO embeddings**: retrieval fine-tuning on the train split + cosine edges
3) **Fine-tuned + EdgeMLP**: fine-tuned embeddings + learned EdgeMLP edge scoring
4) **Everything**: (2)+(3) **plus hyperparameter selection** on the train split, then evaluated on the test split

Metrics reported (per dataset + overall):
- **mAA** (scene recall under greedy scene↔cluster assignment)
- **Clustering Score** (precision under the same assignment)
- **Final Score** (harmonic mean)


In [19]:
from __future__ import annotations

import json
import os
import sys
import shlex
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm


def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "src").exists() and (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root (expected src/ and pyproject.toml)")


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

pio.templates.default = "plotly_white"


def run(cmd: list[str], *, env: dict[str, str] | None = None) -> None:
    print("+", shlex.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=str(REPO_ROOT), env=env)


REPO_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025


## 1) Split `data/train/` into internal train/test

**Test datasets**: `ETs`, `pt_sacrecoeur_trevi_tajmahal`, `pt_stpeters_stpauls`  
**Train datasets**: everything else under `data/train/`.


In [20]:
DATA_TRAIN_ROOT = REPO_ROOT / "data" / "train"
TRAIN_LABELS = REPO_ROOT / "data" / "train_labels.csv"

TEST_DATASETS = [
    "ETs",
    "pt_sacrecoeur_trevi_tajmahal",
    "pt_stpeters_stpauls",
]
TEST_DATASETS_SET = set(TEST_DATASETS)

SPLIT_TAG = "holdout_ets_sacrecoeur_stpeters"
TRAIN_SPLIT_ROOT = REPO_ROOT / "data" / f"train_split_{SPLIT_TAG}"
TEST_SPLIT_ROOT = REPO_ROOT / "data" / f"test_split_{SPLIT_TAG}"

TRAIN_SPLIT_LABELS = REPO_ROOT / "data" / f"train_split_labels_{SPLIT_TAG}.csv"
TEST_SPLIT_LABELS = REPO_ROOT / "data" / f"test_split_labels_{SPLIT_TAG}.csv"

OVERWRITE_SPLIT = False


def _symlink_dir(src: Path, dst: Path, *, overwrite: bool) -> None:
    if dst.exists() or dst.is_symlink():
        if not overwrite:
            return
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            # Directory: remove only if it's inside our split roots.
            for p in sorted(dst.rglob("*"), reverse=True):
                if p.is_symlink() or p.is_file():
                    p.unlink()
                elif p.is_dir():
                    p.rmdir()
            dst.rmdir()
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src, target_is_directory=True)


assert DATA_TRAIN_ROOT.exists(), f"Missing {DATA_TRAIN_ROOT}"
assert TRAIN_LABELS.exists(), f"Missing {TRAIN_LABELS}"

all_datasets = sorted([p.name for p in DATA_TRAIN_ROOT.iterdir() if p.is_dir()])
train_datasets = [d for d in all_datasets if d not in TEST_DATASETS_SET]
test_datasets = [d for d in all_datasets if d in TEST_DATASETS_SET]
print("datasets in data/train:", len(all_datasets))
print("train datasets:", train_datasets)
print("test datasets:", test_datasets)

TRAIN_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
TEST_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
for ds in train_datasets:
    _symlink_dir(DATA_TRAIN_ROOT / ds, TRAIN_SPLIT_ROOT / ds, overwrite=OVERWRITE_SPLIT)
for ds in test_datasets:
    _symlink_dir(DATA_TRAIN_ROOT / ds, TEST_SPLIT_ROOT / ds, overwrite=OVERWRITE_SPLIT)

labels = pd.read_csv(TRAIN_LABELS)
labels_train = labels[labels["dataset"].isin(train_datasets)].copy()
labels_test = labels[labels["dataset"].isin(test_datasets)].copy()

if OVERWRITE_SPLIT or (not TRAIN_SPLIT_LABELS.exists()):
    labels_train.to_csv(TRAIN_SPLIT_LABELS, index=False)
if OVERWRITE_SPLIT or (not TEST_SPLIT_LABELS.exists()):
    labels_test.to_csv(TEST_SPLIT_LABELS, index=False)

(REPO_ROOT / "data" / f"split_info_{SPLIT_TAG}.json").write_text(
    json.dumps(
        {
            "split_tag": SPLIT_TAG,
            "source": str(DATA_TRAIN_ROOT),
            "train_root": str(TRAIN_SPLIT_ROOT),
            "test_root": str(TEST_SPLIT_ROOT),
            "train_labels": str(TRAIN_SPLIT_LABELS),
            "test_labels": str(TEST_SPLIT_LABELS),
            "train_datasets": train_datasets,
            "test_datasets": test_datasets,
        },
        indent=2,
    )
)
print("[ok] split ready")
print("TRAIN_SPLIT_ROOT:", TRAIN_SPLIT_ROOT)
print("TEST_SPLIT_ROOT:", TEST_SPLIT_ROOT)
print("TRAIN_SPLIT_LABELS:", TRAIN_SPLIT_LABELS)
print("TEST_SPLIT_LABELS:", TEST_SPLIT_LABELS)


datasets in data/train: 13
train datasets: ['amy_gardens', 'fbk_vineyard', 'imc2023_haiper', 'imc2023_heritage', 'imc2023_theather_imc2024_church', 'imc2024_dioscuri_baalshamin', 'imc2024_lizard_pond', 'pt_brandenburg_british_buckingham', 'pt_piazzasanmarco_grandplace', 'stairs']
test datasets: ['ETs', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls']
[ok] split ready
TRAIN_SPLIT_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_split_holdout_ets_sacrecoeur_stpeters
TEST_SPLIT_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split_holdout_ets_sacrecoeur_stpeters
TRAIN_SPLIT_LABELS: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_split_labels_holdout_ets_sacrecoeur_stpeters.csv
TEST_SPLIT_LABELS: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split_labels_holdout_ets_sacrecoeur_stpeters.csv


## 2) Scoring utilities (mAA / Clustering Score / Final)

We score each dataset with a greedy **one-to-one** scene↔cluster assignment that prioritizes mAA (ties broken by clustering score).

In [21]:
def _image_keys_in_dir(ds_dir: Path) -> set[str]:
    if not ds_dir.exists():
        return set()
    out = set()
    for p in ds_dir.iterdir():
        if p.is_file() or p.is_symlink():
            if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}:
                out.add(p.stem)
    return out


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem
    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id` to derive an image key")
    s = image_id.strip()
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]
    return Path(s).stem


def load_ground_truth(*, labels_csv: Path, data_root: Path, drop_outliers_scene: bool) -> Tuple[Dict[str, Dict[str, set[str]]], Dict[str, set[str]]]:
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"Ground truth missing columns: {sorted(missing)}")

    gt = gt[["dataset", "scene", "image"]].copy()
    if drop_outliers_scene:
        gt = gt[gt["scene"].astype(str).str.lower() != "outliers"].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, scene), g in gt.groupby(["dataset", "scene"], sort=True):
        scenes.setdefault(str(dataset), {})[str(scene)] = set(g["image_key"].tolist())

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        if not keys:
            keys = set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())
        all_images[dataset] = keys

    return scenes, all_images


def load_predictions(*, pred_csv: Path, expected_images: Dict[str, set[str]], outliers_label: str = "outliers") -> Dict[str, Dict[str, set[str]]]:
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("Predictions CSV must contain a `dataset` column")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("Predictions CSV must contain one of: scene / cluster / cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("Predictions CSV must contain `image` or `image_id`")

    df = df[["dataset", cluster_col] + ([image_col] if image_col else []) + ([image_id_col] if image_id_col else [])].copy()
    df = df.rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r[image_id_col]) else None,
            image=str(r[image_col]) if image_col and pd.notna(r[image_col]) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        kept.append(g[g["image_key"].isin(exp)] if exp else g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        rows_missing.extend({"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing)
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    pred: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        pred.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())
    return pred


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


@dataclass(frozen=True)
class DatasetScore:
    dataset: str
    mAA: float
    clustering_score: float
    final_score: float
    n_scenes: int
    n_images_in_scenes: int
    n_correct: int
    n_cluster_images_summed: int


def score_dataset(*, dataset: str, gt_scenes: Dict[str, set[str]], pred_clusters: Dict[str, set[str]], outliers_label: str = "outliers") -> Tuple[DatasetScore, pd.DataFrame]:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    edges: list[tuple[float, float, int, int, str, str]] = []
    for scene, scene_imgs in gt_scenes.items():
        n_scene = len(scene_imgs)
        if n_scene <= 0:
            continue
        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter <= 0:
                continue
            n_cluster = len(c_imgs)
            maa = inter / n_scene
            cs = inter / n_cluster if n_cluster else 0.0
            edges.append((maa, cs, int(inter), -int(n_cluster), str(scene), str(c_label)))

    edges.sort(key=lambda x: (x[0], x[1], x[2], x[3]), reverse=True)

    assigned_scene: dict[str, tuple[str, int, float, float, int]] = {}
    used_clusters: set[str] = set()
    for maa, cs, inter, neg_n_cluster, scene, c_label in edges:
        if scene in assigned_scene:
            continue
        if c_label in used_clusters:
            continue
        assigned_scene[scene] = (c_label, int(inter), float(maa), float(cs), int(-neg_n_cluster))
        used_clusters.add(c_label)

    rows = []
    total_correct = 0
    total_scene_images = 0
    total_cluster_images = 0

    for scene, scene_imgs in sorted(gt_scenes.items()):
        n_scene = len(scene_imgs)
        total_scene_images += n_scene

        if scene in assigned_scene:
            best_cluster, best_correct, best_maa, best_cs, best_cluster_size = assigned_scene[scene]
        else:
            best_cluster = "(none)"
            best_correct = 0
            best_maa = 0.0
            best_cs = 0.0
            best_cluster_size = 0

        total_correct += int(best_correct)
        total_cluster_images += int(best_cluster_size)

        rows.append(
            {
                "dataset": dataset,
                "gt_scene": scene,
                "assigned_cluster": best_cluster,
                "n_scene": n_scene,
                "n_cluster": int(best_cluster_size),
                "n_correct": int(best_correct),
                "mAA(scene)": float(best_maa),
                "CS(scene)": float(best_cs),
            }
        )

    maa = (total_correct / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    final = harmonic_mean(maa, cs)

    ds_score = DatasetScore(
        dataset=dataset,
        mAA=float(maa),
        clustering_score=float(cs),
        final_score=float(final),
        n_scenes=len(gt_scenes),
        n_images_in_scenes=int(total_scene_images),
        n_correct=int(total_correct),
        n_cluster_images_summed=int(total_cluster_images),
    )

    return ds_score, pd.DataFrame(rows)


def score_predictions_csv(*, pred_csv: Path, labels_csv: Path, data_root: Path, outliers_label: str = "outliers", drop_outliers_scene: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    gt_scenes_by_dataset, all_images_by_dataset = load_ground_truth(labels_csv=labels_csv, data_root=data_root, drop_outliers_scene=drop_outliers_scene)
    pred_clusters_by_dataset = load_predictions(pred_csv=pred_csv, expected_images=all_images_by_dataset, outliers_label=outliers_label)

    datasets = sorted(set(gt_scenes_by_dataset.keys()) & set(pred_clusters_by_dataset.keys()))
    dataset_scores = []
    scene_tables = []

    for ds in datasets:
        ds_score, per_scene = score_dataset(
            dataset=ds,
            gt_scenes=gt_scenes_by_dataset[ds],
            pred_clusters=pred_clusters_by_dataset[ds],
            outliers_label=outliers_label,
        )
        dataset_scores.append(ds_score.__dict__)
        scene_tables.append(per_scene)

    scores_df = pd.DataFrame(dataset_scores).sort_values("dataset")
    scenes_df = pd.concat(scene_tables, ignore_index=True) if scene_tables else pd.DataFrame()

    overall = {
        "datasets": datasets,
        "macro_final_mean": float(scores_df["final_score"].mean()) if len(scores_df) else 0.0,
        "micro_mAA": 0.0,
        "micro_CS": 0.0,
        "micro_final": 0.0,
    }
    if len(scores_df):
        total_correct = int(scores_df["n_correct"].sum())
        total_scene_images = int(scores_df["n_images_in_scenes"].sum())
        total_cluster_images = int(scores_df["n_cluster_images_summed"].sum())
        micro_maa = total_correct / total_scene_images if total_scene_images else 0.0
        micro_cs = total_correct / total_cluster_images if total_cluster_images else 0.0
        overall["micro_mAA"] = float(micro_maa)
        overall["micro_CS"] = float(micro_cs)
        overall["micro_final"] = float(harmonic_mean(micro_maa, micro_cs))

    return scores_df, scenes_df, overall


## 3) Build inference-only (pretrained) embedding cache for the test split

This produces a cache compatible with `scripts/cluster_retrieval.py`:
- `meta.json`
- `embeddings.npy`

No fine-tuning is performed.

In [22]:
from imc25.train.retrieval_transforms import load_transform_spec, make_eval_transform
from transformers import AutoModel


MODEL_ID = "facebook/dinov2-small"

HF_ENDPOINT = None  # "https://hf-mirror.com"

OUT_ROOT = REPO_ROOT / "outputs" / f"ablation_{SPLIT_TAG}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_PRETRAINED_TEST = OUT_ROOT / "cache_retrieval_pretrained_test"
CACHE_PRETRAINED_TEST.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


class _PathDataset(torch.utils.data.Dataset):
    def __init__(self, paths: list[Path], *, transform) -> None:
        self.paths = paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image

        p = self.paths[int(idx)]
        with Image.open(p) as im:
            im = im.convert("RGB")
            x = self.transform(im)
        return x, str(p)


def _list_images(root: Path) -> list[Path]:
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
    out: list[Path] = []
    for p in sorted(root.iterdir()):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(p)
    return out


@torch.inference_mode()
def _embed_pretrained_backbone(paths: list[Path], *, batch_size: int, num_workers: int) -> np.ndarray:
    if HF_ENDPOINT:
        os.environ["HF_ENDPOINT"] = str(HF_ENDPOINT)

    spec = load_transform_spec(model_id=MODEL_ID, hf_endpoint=HF_ENDPOINT)
    tf = make_eval_transform(spec)

    model = AutoModel.from_pretrained(MODEL_ID)
    model.eval().to(DEVICE)

    ds = _PathDataset(paths, transform=tf)
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(num_workers),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

    zs: list[np.ndarray] = []
    for xb, _ in tqdm(loader, desc="embed(pretrained)", leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        out = model(pixel_values=xb)
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            x = out.pooler_output
        else:
            x = out.last_hidden_state[:, 0]
        z = F.normalize(x.float(), p=2, dim=1).cpu().numpy()
        zs.append(z.astype(np.float32, copy=False))
    if not zs:
        return np.zeros((0, 1), dtype=np.float32)
    return np.concatenate(zs, axis=0)


def build_cache_pretrained(*, data_root: Path, cache_root: Path, datasets: list[str], batch_size: int = 64, num_workers: int = 0, overwrite: bool = False) -> None:
    cache_root.mkdir(parents=True, exist_ok=True)
    for dataset in datasets:
        ds_dir = data_root / dataset
        out_dir = cache_root / dataset
        out_dir.mkdir(parents=True, exist_ok=True)
        meta_path = out_dir / "meta.json"
        emb_path = out_dir / "embeddings.npy"
        if meta_path.exists() and emb_path.exists() and not overwrite:
            print(f"[skip] pretrained cache exists: {dataset} -> {out_dir}")
            continue
        paths = _list_images(ds_dir)
        if not paths:
            print(f"[skip] no images: {ds_dir}")
            continue
        image_ids = [p.stem for p in paths]
        emb = _embed_pretrained_backbone(paths, batch_size=batch_size, num_workers=num_workers)
        np.save(emb_path, emb.astype(np.float32, copy=False))
        meta = {
            "dataset": dataset,
            "backend": "pretrained_backbone",
            "model": MODEL_ID,
            "embed_dim": int(emb.shape[1]) if emb.ndim == 2 else None,
            "image_ids": image_ids,
            "paths": [str(p) for p in paths],
        }
        meta_path.write_text(json.dumps(meta, indent=2))
        print(f"[ok] pretrained cache: {dataset} embeddings={emb.shape} -> {out_dir}")


# Build the inference-only cache for the HOLDOUT test datasets.
BUILD_PRETRAINED_CACHE = True
if BUILD_PRETRAINED_CACHE:
    build_cache_pretrained(
        data_root=TEST_SPLIT_ROOT,
        cache_root=CACHE_PRETRAINED_TEST,
        datasets=test_datasets,
        batch_size=64,
        # NOTE: keep num_workers=0 to avoid PyTorch DataLoader multiprocessing shutdown errors.
        num_workers=0,
        overwrite=False,
    )


device: cuda
[skip] pretrained cache exists: ETs -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_pretrained_test/ETs
[skip] pretrained cache exists: pt_sacrecoeur_trevi_tajmahal -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_pretrained_test/pt_sacrecoeur_trevi_tajmahal
[skip] pretrained cache exists: pt_stpeters_stpauls -> /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_pretrained_test/pt_stpeters_stpauls


## 4) Fine-tune retrieval embeddings (DINOv2 backbone + projection head)

This trains on the **train split** only, and produces a checkpoint for configurations (2)–(4).

In [23]:
TRAIN_OUTPUT_DIR = OUT_ROOT / "retrieval_finetune"
CHECKPOINT_FINETUNED = TRAIN_OUTPUT_DIR / "best.pt"

RUN_TRAINING = False  # set True if you want to (re)train now

env = os.environ.copy()
if HF_ENDPOINT:
    env["HF_ENDPOINT"] = str(HF_ENDPOINT)

if RUN_TRAINING or (not CHECKPOINT_FINETUNED.exists()):
    cmd = [
        sys.executable,
        "scripts/train_retrieval.py",
        "--data-root",
        str(TRAIN_SPLIT_ROOT),
        "--labels-csv",
        str(TRAIN_SPLIT_LABELS),
        "--output-dir",
        str(TRAIN_OUTPUT_DIR),
        "--model-id",
        str(MODEL_ID),
        "--embed-dim",
        "256",
        "--epochs",
        "6",
        "--batch-size",
        "48",
        "--num-workers",
        "4",
        "--freeze-backbone",
        "--no-include-outliers",
    ]
    run(cmd, env=env)
else:
    print("[skip] training:", CHECKPOINT_FINETUNED)

assert CHECKPOINT_FINETUNED.exists(), f"Missing checkpoint: {CHECKPOINT_FINETUNED}"
CHECKPOINT_FINETUNED


[skip] training: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/retrieval_finetune/best.pt


PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/retrieval_finetune/best.pt')

## 5) Build fine-tuned embedding caches (train + test split)

These caches are used for clustering and for training the EdgeMLP.

In [24]:
import sys

CACHE_FINETUNED_TRAIN = OUT_ROOT / "cache_retrieval_finetuned_train"
CACHE_FINETUNED_TEST = OUT_ROOT / "cache_retrieval_finetuned_test"

BUILD_FINETUNED_CACHES = True

if BUILD_FINETUNED_CACHES:
    # Train cache (for EdgeMLP training + hyperparam selection)
    cmd = [
        sys.executable,
        "scripts/build_retrieval_cache.py",
        "--data-root",
        str(TRAIN_SPLIT_ROOT),
        "--cache-root",
        str(CACHE_FINETUNED_TRAIN),
        "--checkpoint",
        str(CHECKPOINT_FINETUNED),
        "--topk",
        "30",
        "--mutual",
    ]
    if HF_ENDPOINT:
        cmd += ["--hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=env)

    # Test cache (for evaluation)
    cmd = [
        sys.executable,
        "scripts/build_retrieval_cache.py",
        "--data-root",
        str(TEST_SPLIT_ROOT),
        "--cache-root",
        str(CACHE_FINETUNED_TEST),
        "--checkpoint",
        str(CHECKPOINT_FINETUNED),
        "--topk",
        "30",
        "--mutual",
    ]
    if HF_ENDPOINT:
        cmd += ["--hf-endpoint", str(HF_ENDPOINT)]
    run(cmd, env=env)


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/build_retrieval_cache.py --data-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_split_holdout_ets_sacrecoeur_stpeters --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_finetuned_train --checkpoint /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/retrieval_finetune/best.pt --topk 30 --mutual


Traceback (most recent call last):
  File "/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/scripts/build_retrieval_cache.py", line 12, in <module>
    from imc25.train.retrieval_cache import main
  File "/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/src/imc25/train/retrieval_cache.py", line 9, in <module>
    import torch
  File "/home/shamika/miniconda3/envs/imc25/lib/python3.10/site-packages/torch/__init__.py", line 2161, in <module>
    from torch import _VF as _VF, functional as functional  # usort: skip
  File "/home/shamika/miniconda3/envs/imc25/lib/python3.10/site-packages/torch/functional.py", line 8, in <module>
    import torch.nn.functional as F
  File "/home/shamika/miniconda3/envs/imc25/lib/python3.10/site-packages/torch/nn/__init__.py", line 9, in <module>
    from torch.nn import (
  File "/home/shamika/miniconda3/envs/imc25/lib/python3.10/site-packages/torch/nn/attention/__init__.py", line 9, in <module>
    import

KeyboardInterrupt: 

## 6) Train EdgeMLP on the train split (using fine-tuned embeddings)

This trains a lightweight classifier to score candidate KNN edges as "same-scene" vs "different-scene".

The saved checkpoint is then used in configurations (3)–(4).

In [ ]:
from imc25.models.edge_mlp import EdgeMLP

EDGE_MODELS_DIR = OUT_ROOT / "edge_models"
EDGE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
EDGE_MLP_CKPT = EDGE_MODELS_DIR / "edge_mlp_split.pt"

RUN_EDGE_MLP_TRAINING = False  # set True to force retraining

EDGE_K = 30
EDGE_VAL_RATIO = 0.1
EDGE_EPOCHS = 12
EDGE_BATCH_SIZE = 2048
EDGE_LR = 1e-3
EDGE_SEED = 42


def _normalize_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    denom = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / denom


def _load_cache_embeddings(cache_root: Path, dataset: str) -> tuple[list[str], np.ndarray]:
    meta = json.loads((cache_root / dataset / "meta.json").read_text())
    image_ids = [str(x) for x in meta.get("image_ids", [])]
    emb = np.load(cache_root / dataset / "embeddings.npy")
    if len(image_ids) != int(emb.shape[0]):
        raise ValueError(f"{dataset}: meta/image_ids mismatch embeddings rows")
    return image_ids, emb.astype(np.float32, copy=False)


def _split_indices(n: int, *, val_ratio: float, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(int(seed))
    idx = np.arange(int(n))
    rng.shuffle(idx)
    n_val = int(round(float(val_ratio) * int(n)))
    val = np.sort(idx[:n_val])
    tr = np.sort(idx[n_val:])
    return tr, val


def _build_edge_dataset_for_dataset(
    *,
    dataset: str,
    cache_root: Path,
    labels_csv: Path,
    k: int,
    treat_outliers_as_negative: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return (i, j, y, emb) for one dataset, where i/j index into emb rows."""
    image_ids, emb = _load_cache_embeddings(cache_root, dataset)
    emb = _normalize_rows(emb)

    gt = pd.read_csv(labels_csv)
    gt = gt[gt["dataset"] == dataset].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)
    scene_of = {str(r.image_key): str(r.scene) for r in gt.itertuples(index=False)}
    scene = [scene_of.get(str(i), "(missing)") for i in image_ids]

    n = int(emb.shape[0])
    if n <= 1:
        return np.zeros((0,), np.int32), np.zeros((0,), np.int32), np.zeros((0,), np.float32), emb

    kk = int(min(int(k) + 1, n))
    nn = NearestNeighbors(n_neighbors=kk, metric="cosine")
    nn.fit(emb)
    _, ind = nn.kneighbors(emb, return_distance=True)
    neigh = ind[:, 1:]

    i_list: list[int] = []
    j_list: list[int] = []
    y_list: list[float] = []
    for i in range(n):
        si = scene[i]
        for j in neigh[i].tolist():
            j = int(j)
            if i == j:
                continue
            sj = scene[j]
            if treat_outliers_as_negative and (si.lower() == "outliers" or sj.lower() == "outliers"):
                y = 0.0
            else:
                y = 1.0 if si == sj else 0.0
            i_list.append(i)
            j_list.append(j)
            y_list.append(float(y))

    return (
        np.asarray(i_list, dtype=np.int32),
        np.asarray(j_list, dtype=np.int32),
        np.asarray(y_list, dtype=np.float32),
        emb,
    )


def train_edge_mlp(*, cache_root: Path, labels_csv: Path, datasets: list[str], out_path: Path) -> Path:
    # Gather all edges across datasets.
    all_i: list[np.ndarray] = []
    all_j: list[np.ndarray] = []
    all_y: list[np.ndarray] = []
    all_emb: list[np.ndarray] = []
    all_offsets: list[int] = []

    offset = 0
    for ds in datasets:
        i, j, y, emb = _build_edge_dataset_for_dataset(dataset=ds, cache_root=cache_root, labels_csv=labels_csv, k=EDGE_K)
        all_offsets.append(offset)
        all_i.append(i + offset)
        all_j.append(j + offset)
        all_y.append(y)
        all_emb.append(emb)
        offset += int(emb.shape[0])

    Z = np.concatenate(all_emb, axis=0) if all_emb else np.zeros((0, 256), dtype=np.float32)
    I = np.concatenate(all_i, axis=0) if all_i else np.zeros((0,), dtype=np.int32)
    J = np.concatenate(all_j, axis=0) if all_j else np.zeros((0,), dtype=np.int32)
    Y = np.concatenate(all_y, axis=0) if all_y else np.zeros((0,), dtype=np.float32)

    if len(Z) == 0 or len(I) == 0:
        raise RuntimeError("No edges/embeddings available to train EdgeMLP")

    # Split by embedding index (so we don't leak anchors between train/val).
    tr_idx, va_idx = _split_indices(len(Z), val_ratio=EDGE_VAL_RATIO, seed=EDGE_SEED)
    tr_set = set(tr_idx.tolist())
    va_set = set(va_idx.tolist())

    tr_mask = np.array([(int(i) in tr_set and int(j) in tr_set) for i, j in zip(I, J, strict=False)], dtype=bool)
    va_mask = np.array([(int(i) in va_set and int(j) in va_set) for i, j in zip(I, J, strict=False)], dtype=bool)

    I_tr, J_tr, Y_tr = I[tr_mask], J[tr_mask], Y[tr_mask]
    I_va, J_va, Y_va = I[va_mask], J[va_mask], Y[va_mask]

    print(f"EdgeMLP training edges: train={len(I_tr)} val={len(I_va)} (total={len(I)})")

    z_t = torch.from_numpy(Z).to(DEVICE)
    model = EdgeMLP(int(Z.shape[1])).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=float(EDGE_LR))
    loss_fn = nn.BCELoss()

    best_auc = -1.0
    best_state = None

    rng = np.random.default_rng(EDGE_SEED)
    for epoch in range(1, int(EDGE_EPOCHS) + 1):
        model.train()
        order = rng.permutation(len(I_tr))
        losses = []
        for start in range(0, len(order), int(EDGE_BATCH_SIZE)):
            idx = order[start : start + int(EDGE_BATCH_SIZE)]
            ii = torch.from_numpy(I_tr[idx]).to(DEVICE)
            jj = torch.from_numpy(J_tr[idx]).to(DEVICE)
            yb = torch.from_numpy(Y_tr[idx]).to(DEVICE)

            opt.zero_grad(set_to_none=True)
            pred = model(z_t[ii], z_t[jj])
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu().item()))

        model.eval()
        with torch.no_grad():
            if len(I_va) > 0:
                ii = torch.from_numpy(I_va).to(DEVICE)
                jj = torch.from_numpy(J_va).to(DEVICE)
                yv = Y_va
                pv = model(z_t[ii], z_t[jj]).detach().cpu().numpy()
                auc = float(roc_auc_score(yv, pv)) if len(np.unique(yv)) > 1 else 0.0
                ap = float(average_precision_score(yv, pv)) if len(np.unique(yv)) > 1 else 0.0
            else:
                auc, ap = 0.0, 0.0

        print(f"Epoch {epoch:02d} | loss={np.mean(losses) if losses else 0:.4f} | val_auc={auc:.4f} | val_ap={ap:.4f}")
        if auc >= best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        best_state = model.state_dict()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": best_state, "dim": int(Z.shape[1]), "K": int(EDGE_K)}, out_path)
    print(f"[ok] saved EdgeMLP: {out_path} (best_auc={best_auc:.4f})")
    return out_path


if RUN_EDGE_MLP_TRAINING or (not EDGE_MLP_CKPT.exists()):
    train_edge_mlp(cache_root=CACHE_FINETUNED_TRAIN, labels_csv=TRAIN_SPLIT_LABELS, datasets=train_datasets, out_path=EDGE_MLP_CKPT)
else:
    print("[skip] EdgeMLP exists:", EDGE_MLP_CKPT)

assert EDGE_MLP_CKPT.exists(), f"Missing EdgeMLP checkpoint: {EDGE_MLP_CKPT}"
EDGE_MLP_CKPT


EdgeMLP training edges: train=36455 val=473 (total=44940)
Epoch 01 | loss=0.6312 | val_auc=0.3228 | val_ap=0.8030
Epoch 02 | loss=0.5001 | val_auc=0.3450 | val_ap=0.8079
Epoch 03 | loss=0.4330 | val_auc=0.7933 | val_ap=0.9677
Epoch 04 | loss=0.3620 | val_auc=0.8196 | val_ap=0.9737
Epoch 05 | loss=0.3169 | val_auc=0.8742 | val_ap=0.9833
Epoch 06 | loss=0.2747 | val_auc=0.9289 | val_ap=0.9912
Epoch 07 | loss=0.2272 | val_auc=0.9724 | val_ap=0.9967
Epoch 08 | loss=0.1853 | val_auc=0.9817 | val_ap=0.9978
Epoch 09 | loss=0.1596 | val_auc=0.9826 | val_ap=0.9979
Epoch 10 | loss=0.1446 | val_auc=0.9848 | val_ap=0.9982
Epoch 11 | loss=0.1348 | val_auc=0.9854 | val_ap=0.9983
Epoch 12 | loss=0.1278 | val_auc=0.9862 | val_ap=0.9984
[ok] saved EdgeMLP: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp_split.pt (best_auc=0.9862)


PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp_split.pt')

## 7) Run the 4 configurations (cluster → score → visualize)

We keep the evaluation split fixed, and compare different components.


In [ ]:
@dataclass(frozen=True)
class ClusterCfg:
    topk: int = 30
    mutual: bool = True
    min_sim: float = 0.35
    method: str = "louvain"  # louvain|greedy|components
    min_cluster_size: int = 3
    edge_scorer: str = "cosine"  # cosine|mlp|gbdt
    edge_model: Path | None = None
    edge_topm: int | None = None


def run_cluster_retrieval(*, cache_root: Path, out_csv: Path, datasets: list[str], cfg: ClusterCfg) -> None:
    cmd = [
        sys.executable,
        "scripts/cluster_retrieval.py",
        "--cache-root",
        str(cache_root),
        "--out-csv",
        str(out_csv),
        "--topk",
        str(int(cfg.topk)),
        "--min-sim",
        str(float(cfg.min_sim)),
        "--method",
        str(cfg.method),
        "--min-cluster-size",
        str(int(cfg.min_cluster_size)),
        "--edge-scorer",
        str(cfg.edge_scorer),
    ]
    for ds in datasets:
        cmd += ["--dataset", str(ds)]

    if cfg.edge_scorer in {"mlp", "gbdt"}:
        if cfg.edge_model is None:
            raise ValueError("edge_model required when edge_scorer is mlp/gbdt")
        cmd += ["--edge-model", str(cfg.edge_model)]
        if cfg.edge_topm is not None:
            cmd += ["--edge-topm", str(int(cfg.edge_topm))]

    if cfg.mutual:
        cmd += ["--mutual"]
    else:
        cmd += ["--no-mutual"]

    run(cmd, env=env)


DROP_GT_OUTLIERS = True  # set False to include GT outliers scene in scoring
OUTLIERS_LABEL = "outliers"

RESULTS_DIR = OUT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def eval_clusters(*, name: str, clusters_csv: Path) -> dict:
    scores_df, scenes_df, overall = score_predictions_csv(
        pred_csv=clusters_csv,
        labels_csv=TEST_SPLIT_LABELS,
        data_root=TEST_SPLIT_ROOT,
        outliers_label=OUTLIERS_LABEL,
        drop_outliers_scene=DROP_GT_OUTLIERS,
    )
    scores_df.to_csv(RESULTS_DIR / f"{name}_dataset_scores.csv", index=False)
    scenes_df.to_csv(RESULTS_DIR / f"{name}_per_scene.csv", index=False)
    (RESULTS_DIR / f"{name}_overall.json").write_text(json.dumps(overall, indent=2))
    return {
        "name": name,
        "scores_df": scores_df,
        "scenes_df": scenes_df,
        "overall": overall,
    }


# --- Config 1: inference-only (pretrained backbone) ---
CFG1 = ClusterCfg(edge_scorer="cosine", min_sim=0.35, topk=30, mutual=True, method="louvain")
CLUSTERS1 = RESULTS_DIR / "clusters_cfg1_inference_only.csv"
run_cluster_retrieval(cache_root=CACHE_PRETRAINED_TEST, out_csv=CLUSTERS1, datasets=test_datasets, cfg=CFG1)
res1 = eval_clusters(name="cfg1_inference_only", clusters_csv=CLUSTERS1)

# --- Config 2: finetuned embeddings + cosine ---
CFG2 = ClusterCfg(edge_scorer="cosine", min_sim=0.35, topk=30, mutual=True, method="louvain")
CLUSTERS2 = RESULTS_DIR / "clusters_cfg2_finetuned_cosine.csv"
run_cluster_retrieval(cache_root=CACHE_FINETUNED_TEST, out_csv=CLUSTERS2, datasets=test_datasets, cfg=CFG2)
res2 = eval_clusters(name="cfg2_finetuned_cosine", clusters_csv=CLUSTERS2)

# --- Config 3: finetuned embeddings + EdgeMLP (fixed hyperparams) ---
CFG3 = ClusterCfg(edge_scorer="mlp", edge_model=EDGE_MLP_CKPT, edge_topm=5, min_sim=0.55, topk=30, mutual=True, method="louvain")
CLUSTERS3 = RESULTS_DIR / "clusters_cfg3_finetuned_edge_mlp.csv"
run_cluster_retrieval(cache_root=CACHE_FINETUNED_TEST, out_csv=CLUSTERS3, datasets=test_datasets, cfg=CFG3)
res3 = eval_clusters(name="cfg3_finetuned_edge_mlp", clusters_csv=CLUSTERS3)


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/cluster_retrieval.py --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_pretrained_test --out-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/results/clusters_cfg1_inference_only.csv --topk 30 --min-sim 0.35 --method louvain --min-cluster-size 3 --edge-scorer cosine --dataset ETs --dataset pt_sacrecoeur_trevi_tajmahal --dataset pt_stpeters_stpauls --mutual
Inference embeddings shape: (22, 384)
[ok] ETs: clusters=2 outliers=0 images=22
Inference embeddings shape: (225, 384)
[ok] pt_sacrecoeur_trevi_tajmahal: clusters=6 outliers=4 images=225
Inference embeddings shape: (200, 384)
[ok] pt_stpeters_stpauls: clusters=7 outliers=3 images=200
[ok] wrote /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sa

## 8) Config 4 ("everything"): select clustering hyperparameters on the train split, then evaluate on the test split

We do a small grid search using **train split** ground truth, pick the best config by overall micro Final Score, and then run it on the holdout test datasets.

In [ ]:
SWEEP = True

sweep_min_sim = [0.35, 0.45, 0.55, 0.65]
sweep_edge_topm = [None, 10, 5, 3]
sweep_method = ["louvain", "greedy"]

SWEEP_OUT = RESULTS_DIR / "sweep_cfg4_train_selection.csv"


def score_on_train_split(*, clusters_csv: Path) -> dict:
    scores_df, _scenes_df, overall = score_predictions_csv(
        pred_csv=clusters_csv,
        labels_csv=TRAIN_SPLIT_LABELS,
        data_root=TRAIN_SPLIT_ROOT,
        outliers_label=OUTLIERS_LABEL,
        drop_outliers_scene=DROP_GT_OUTLIERS,
    )
    return {
        "macro_final_mean": float(overall["macro_final_mean"]),
        "micro_mAA": float(overall["micro_mAA"]),
        "micro_CS": float(overall["micro_CS"]),
        "micro_final": float(overall["micro_final"]),
        "n_datasets": int(len(overall["datasets"])),
    }


best_cfg4: ClusterCfg | None = None
best_score = -1.0
rows = []

if SWEEP:
    for method in sweep_method:
        for min_sim in sweep_min_sim:
            for topm in sweep_edge_topm:
                cfg = ClusterCfg(
                    edge_scorer="mlp",
                    edge_model=EDGE_MLP_CKPT,
                    edge_topm=topm,
                    min_sim=float(min_sim),
                    topk=30,
                    mutual=True,
                    method=str(method),
                    min_cluster_size=3,
                )
                out_csv = RESULTS_DIR / f"tmp_cfg4_train_{method}_mins{min_sim}_topm{topm}.csv"
                run_cluster_retrieval(cache_root=CACHE_FINETUNED_TRAIN, out_csv=out_csv, datasets=train_datasets, cfg=cfg)
                m = score_on_train_split(clusters_csv=out_csv)
                row = {
                    "method": method,
                    "min_sim": float(min_sim),
                    "edge_topm": None if topm is None else int(topm),
                    **m,
                }
                rows.append(row)
                if float(m["micro_final"]) > best_score:
                    best_score = float(m["micro_final"])
                    best_cfg4 = cfg

    sweep_df = pd.DataFrame(rows).sort_values(["micro_final"], ascending=False)
    sweep_df.to_csv(SWEEP_OUT, index=False)
    print("[ok] sweep written:", SWEEP_OUT)
    print("best micro_final (train):", best_score)
else:
    # A reasonable default if you don't want to sweep.
    best_cfg4 = ClusterCfg(edge_scorer="mlp", edge_model=EDGE_MLP_CKPT, edge_topm=5, min_sim=0.55, topk=30, mutual=True, method="louvain")

assert best_cfg4 is not None
best_cfg4


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/cluster_retrieval.py --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_finetuned_train --out-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/results/tmp_cfg4_train_louvain_mins0.35_topmNone.csv --topk 30 --min-sim 0.35 --method louvain --min-cluster-size 3 --edge-scorer mlp --dataset amy_gardens --dataset fbk_vineyard --dataset imc2023_haiper --dataset imc2023_heritage --dataset imc2023_theather_imc2024_church --dataset imc2024_dioscuri_baalshamin --dataset imc2024_lizard_pond --dataset pt_brandenburg_british_buckingham --dataset pt_piazzasanmarco_grandplace --dataset stairs --edge-model /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp_split.pt --mutual
In

ClusterCfg(topk=30, mutual=True, min_sim=0.35, method='greedy', min_cluster_size=3, edge_scorer='mlp', edge_model=PosixPath('/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp_split.pt'), edge_topm=None)

In [ ]:
# Run config 4 on the holdout test datasets using the selected hyperparams.
CFG4 = best_cfg4
CLUSTERS4 = RESULTS_DIR / "clusters_cfg4_everything_selected.csv"
run_cluster_retrieval(cache_root=CACHE_FINETUNED_TEST, out_csv=CLUSTERS4, datasets=test_datasets, cfg=CFG4)
res4 = eval_clusters(name="cfg4_everything_selected", clusters_csv=CLUSTERS4)

res4["overall"]


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/cluster_retrieval.py --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_finetuned_test --out-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/results/clusters_cfg4_everything_selected.csv --topk 30 --min-sim 0.35 --method greedy --min-cluster-size 3 --edge-scorer mlp --dataset ETs --dataset pt_sacrecoeur_trevi_tajmahal --dataset pt_stpeters_stpauls --edge-model /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/edge_models/edge_mlp_split.pt --mutual
Inference embeddings shape: (22, 256)
[ok] ETs: clusters=3 outliers=0 images=22
Inference embeddings shape: (225, 256)
[ok] pt_sacrecoeur_trevi_tajmahal: clusters=3 outliers=1 images=225
Inference embeddings shape: (200, 256)
[ok] pt_s

{'datasets': ['ETs', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls'],
 'macro_final_mean': 0.9345278797728686,
 'micro_mAA': 0.9864864864864865,
 'micro_CS': 0.9909502262443439,
 'micro_final': 0.9887133182844244}

## 9) Aggregate results + visuals

In [ ]:
def _stack_scores(name: str, res: dict) -> pd.DataFrame:
    df = res["scores_df"].copy()
    df.insert(0, "config", name)
    return df[["config", "dataset", "mAA", "clustering_score", "final_score", "n_images_in_scenes", "n_correct"]]


scores_all = pd.concat(
    [
        _stack_scores("1) inference-only", res1),
        _stack_scores("2) finetuned DINO", res2),
        _stack_scores("3) finetuned + EdgeMLP", res3),
        _stack_scores("4) everything (selected)", res4),
    ],
    ignore_index=True,
)
scores_all.to_csv(RESULTS_DIR / "ablation_scores_by_dataset.csv", index=False)
scores_all


,config,dataset,mAA,clustering_score,final_score,n_images_in_scenes,n_correct
0,1) inference-only,ETs,1.000000,0.863636,0.926829,19,19
1,1) inference-only,pt_sacrecoeur_trevi_tajmahal,0.675556,1.000000,0.806366,225,152
2,1) inference-only,pt_stpeters_stpauls,0.535000,1.000000,0.697068,200,107
3,2) finetuned DINO,ETs,1.000000,0.863636,0.926829,19,19
4,2) finetuned DINO,pt_sacrecoeur_trevi_tajmahal,0.715556,1.000000,0.834197,225,161
5,2) finetuned DINO,pt_stpeters_stpauls,0.580000,1.000000,0.734177,200,116
6,3) finetuned + EdgeMLP,ETs,0.789474,0.833333,0.810811,19,15
7,3) finetuned + EdgeMLP,pt_sacrecoeur_trevi_tajmahal,0.626667,1.000000,0.770492,225,141
8,3) finetuned + EdgeMLP,pt_stpeters_stpauls,0.315000,1.000000,0.479087,200,63
9,4) everything (selected),ETs,0.789474,0.833333,0.810811,19,15


In [ ]:
# Final score per dataset (grouped bars)
fig = px.bar(
    scores_all,
    x="dataset",
    y="final_score",
    color="config",
    barmode="group",
    title=f"Final Score by dataset ({SPLIT_TAG})",
)
fig.update_layout(xaxis_title="Dataset", yaxis_title="Final Score")
fig.show()


In [ ]:
# Final score per dataset (grouped bars)
fig = px.bar(
    scores_all,
    x="dataset",
    y="final_score",
    color="config",
    barmode="group",
    title=f"Final Score by dataset",
)
fig.update_layout(xaxis_title="Dataset", yaxis_title="Final Score")
fig.show()


In [ ]:
# mAA vs Clustering Score tradeoff
fig = px.scatter(
    scores_all,
    x="mAA",
    y="clustering_score",
    color="config",
    symbol="dataset",
    hover_data=["final_score", "n_correct", "n_images_in_scenes"],
    title="mAA vs Clustering Score",
)
fig.update_layout(xaxis_title="mAA (recall)", yaxis_title="Clustering Score (precision)")
fig.show()


In [ ]:
# Overall micro scores (one bar per config)
overall_rows = []
for name, res in [
    ("1) inference-only", res1),
    ("2) finetuned DINO", res2),
    ("3) finetuned + EdgeMLP", res3),
    ("4) everything (selected)", res4),
]:
    overall_rows.append(
        {
            "config": name,
            "micro_mAA": float(res["overall"]["micro_mAA"]),
            "micro_CS": float(res["overall"]["micro_CS"]),
            "micro_final": float(res["overall"]["micro_final"]),
        }
    )

overall_df = pd.DataFrame(overall_rows)
overall_df.to_csv(RESULTS_DIR / "ablation_overall_micro.csv", index=False)
fig = px.bar(overall_df, x="config", y="micro_final", title="Overall micro Final Score")
fig.update_layout(xaxis_title="Config", yaxis_title="micro Final")
fig.show()
overall_df


,config,micro_mAA,micro_CS,micro_final
0,1) inference-only,0.626126,0.989324,0.766897
1,2) finetuned DINO,0.666667,0.989967,0.796770
2,3) finetuned + EdgeMLP,0.493243,0.986486,0.657658
3,4) everything (selected),0.986486,0.990950,0.988713


In [ ]:
# If you ran the sweep: visualize the sweep surface (train split selection)
if SWEEP and SWEEP_OUT.exists():
    sweep_df = pd.read_csv(SWEEP_OUT)
    sweep_df["edge_topm"] = sweep_df["edge_topm"].fillna(-1).astype(int)
    sweep_df["edge_topm_label"] = sweep_df["edge_topm"].map(lambda x: "None" if x < 0 else str(x))
    fig = px.density_heatmap(
        sweep_df,
        x="min_sim",
        y="edge_topm_label",
        z="micro_final",
        facet_col="method",
        histfunc="avg",
        title="Train split sweep: micro Final (avg)",
    )
    fig.update_layout(xaxis_title="min_sim", yaxis_title="edge_topm")
    fig.show()
else:
    print("(skip) sweep not run")


## 10) Extra visuals (cluster sizes + GT↔cluster overlap heatmaps)

In [ ]:
def cluster_size_table(*, pred_csv: Path, config_name: str) -> pd.DataFrame:
    df = pd.read_csv(pred_csv)
    df = df.rename(columns={"scene": "pred_cluster"})
    df["pred_cluster"] = df["pred_cluster"].astype(str)
    df = df[df["pred_cluster"].str.lower() != OUTLIERS_LABEL.lower()].copy()
    out = (
        df.groupby(["dataset", "pred_cluster"], as_index=False)
        .size()
        .rename(columns={"size": "cluster_size"})
    )
    out.insert(0, "config", config_name)
    return out


sizes = pd.concat(
    [
        cluster_size_table(pred_csv=CLUSTERS1, config_name="1) inference-only"),
        cluster_size_table(pred_csv=CLUSTERS2, config_name="2) finetuned DINO"),
        cluster_size_table(pred_csv=CLUSTERS3, config_name="3) finetuned + EdgeMLP"),
        cluster_size_table(pred_csv=CLUSTERS4, config_name="4) everything (selected)"),
    ],
    ignore_index=True,
)

fig = px.box(
    sizes,
    x="dataset",
    y="cluster_size",
    color="config",
    points="all",
    title="Predicted cluster size distribution (excluding outliers)",
)
fig.update_layout(xaxis_title="Dataset", yaxis_title="Cluster size")
fig.show()

sizes.head()


,config,dataset,pred_cluster,cluster_size
0,1) inference-only,ETs,cluster_0001,12
1,1) inference-only,ETs,cluster_0002,10
2,1) inference-only,pt_sacrecoeur_trevi_tajmahal,cluster_0001,59
3,1) inference-only,pt_sacrecoeur_trevi_tajmahal,cluster_0002,51
4,1) inference-only,pt_sacrecoeur_trevi_tajmahal,cluster_0003,42


In [ ]:
def overlap_heatmap(*, dataset: str, pred_csv: Path, config_name: str) -> None:
    gt = pd.read_csv(TEST_SPLIT_LABELS)
    gt = gt[gt["dataset"] == dataset].copy()
    if DROP_GT_OUTLIERS:
        gt = gt[gt["scene"].astype(str).str.lower() != OUTLIERS_LABEL.lower()].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    gt_scenes = {
        str(scene): set(g["image_key"].tolist()) for scene, g in gt.groupby("scene", sort=True)
    }

    pred = pd.read_csv(pred_csv)
    pred = pred[pred["dataset"] == dataset].copy()
    pred["pred_cluster"] = pred["scene"].astype(str)
    pred["image_key"] = pred["image_id"].map(lambda x: Path(str(x)).stem)
    pred = pred[pred["pred_cluster"].str.lower() != OUTLIERS_LABEL.lower()].copy()

    pred_clusters = {
        str(c): set(g["image_key"].tolist()) for c, g in pred.groupby("pred_cluster", sort=True)
    }

    # Sort clusters by size for readability.
    clusters_sorted = sorted(pred_clusters.keys(), key=lambda c: (-len(pred_clusters[c]), c))
    scenes_sorted = sorted(gt_scenes.keys())

    mat = []
    for s in scenes_sorted:
        row = []
        for c in clusters_sorted:
            row.append(len(gt_scenes[s] & pred_clusters[c]))
        mat.append(row)

    dfm = pd.DataFrame(mat, index=scenes_sorted, columns=clusters_sorted)
    fig = px.imshow(
        dfm,
        text_auto=True,
        aspect="auto",
        title=f"{dataset}: GT scenes vs predicted clusters ({config_name})",
    )
    fig.update_layout(xaxis_title="Predicted cluster", yaxis_title="GT scene")
    fig.show()


SHOW_HEATMAPS = True
if SHOW_HEATMAPS:
    for ds in test_datasets:
        overlap_heatmap(dataset=ds, pred_csv=CLUSTERS1, config_name="1) inference-only")
        overlap_heatmap(dataset=ds, pred_csv=CLUSTERS2, config_name="2) finetuned DINO")
        overlap_heatmap(dataset=ds, pred_csv=CLUSTERS3, config_name="3) finetuned + EdgeMLP")
        overlap_heatmap(dataset=ds, pred_csv=CLUSTERS4, config_name="4) everything (selected)")


## 11) SfM matcher comparison (COLMAP / DINO / Diffusion / LightGlue)

Optional: run sparse SfM on the **same clusters** using different matchers.
- COLMAP: classic SIFT matcher
- DINO: patch-descriptor matcher
- Diffusion: diffusion-feature matcher
- LightGlue: ALIKED + LightGlue matcher (optional dependency)

Notes:
- This section runs **sparse-only** SfM (faster).
- LightGlue requires the `lightglue` package + weights.
- Results are saved under `outputs/ablation_<split>/sfm_matcher_compare/`.


In [25]:
import shutil

SFM_OUT_ROOT = OUT_ROOT / "sfm_matcher_compare"
SFM_OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Use the selected clustering (cfg4 by default).
CLUSTERS_FOR_SFM = CLUSTERS4

RUN_SFM_MATCHERS = True  # set True to run COLMAP/DINO/DIFFUSION
RUN_LIGHTGLUE = True  # set True to run ALIKED+LightGlue matcher

RUNNER = "auto"  # auto|local|docker (for colmap_dense_clusters)
OVERWRITE_SFM = False

LIGHTGLUE_CACHE = REPO_ROOT / "cache" / "lightglue_features"
LIGHTGLUE_MAX_KPTS = 2048
LIGHTGLUE_PAIRING = "auto"  # auto|exhaustive|topk
LIGHTGLUE_TOPK = 10
LIGHTGLUE_MIN_INLIERS = 15

def run_recon_matcher(*, matcher: str, out_root: Path) -> None:
    cmd = [
        sys.executable,
        "scripts/colmap_dense_clusters.py",
        "--clusters-csv",
        str(CLUSTERS_FOR_SFM),
        "--cache-root",
        str(CACHE_FINETUNED_TEST),
        "--data-root",
        str(TEST_SPLIT_ROOT),
        "--output-root",
        str(out_root),
        "--runner",
        str(RUNNER),
        "--matcher",
        str(matcher),
        "--sparse-only",
    ]
    if OVERWRITE_SFM:
        cmd += ["--overwrite"]
    run(cmd, env=env)

def _load_meta_map(cache_root: Path, dataset: str) -> dict[str, Path]:
    meta_path = cache_root / dataset / "meta.json"
    if not meta_path.exists():
        return {}
    meta = json.loads(meta_path.read_text())
    image_ids = meta.get("image_ids", [])
    paths = meta.get("paths", [])
    if not isinstance(image_ids, list) or not isinstance(paths, list) or len(image_ids) != len(paths):
        return {}
    out: dict[str, Path] = {}
    for image_id, p in zip(image_ids, paths, strict=False):
        out[str(image_id)] = Path(str(p))
    return out

def _resolve_image_path(*, dataset: str, image_id: str, meta_map: dict[str, Path]) -> Path | None:
    if image_id in meta_map:
        p = meta_map[image_id]
        p = p if p.is_absolute() else (REPO_ROOT / p)
        return p if p.exists() else None
    base = TEST_SPLIT_ROOT / dataset / image_id
    for ext in (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"):
        p = base.with_suffix(ext)
        if p.exists():
            return p
    return None

def _prepare_images_dir(*, dataset: str, scene: str, image_ids: list[str], out_dir: Path, meta_map: dict[str, Path]) -> Path:
    images_dir = out_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    desired = set()
    for image_id in image_ids:
        p = _resolve_image_path(dataset=dataset, image_id=str(image_id), meta_map=meta_map)
        if p is None:
            print(f"[warn] missing image {dataset}/{scene}: {image_id}")
            continue
        dst = images_dir / p.name
        desired.add(dst.name)
        if not dst.exists():
            shutil.copy2(p, dst)
    for entry in images_dir.iterdir():
        if entry.is_file() and entry.name not in desired:
            entry.unlink(missing_ok=True)
    return images_dir

def run_lightglue_sfm(*, clusters_csv: Path, out_root: Path) -> None:
    import pandas as pd
    clusters = pd.read_csv(clusters_csv)
    clusters = clusters[clusters["scene"].astype(str).str.lower() != "outliers"].copy()

    for (dataset, scene), g in clusters.groupby(["dataset", "scene"], sort=True):
        image_ids = g["image_id"].astype(str).tolist()
        if len(image_ids) < 2:
            continue
        out_dir = out_root / f"{dataset}_{scene}"
        if out_dir.exists() and not OVERWRITE_SFM:
            print(f"[skip] lightglue {dataset}/{scene}: exists")
            continue
        if OVERWRITE_SFM and out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        meta_map = _load_meta_map(CACHE_FINETUNED_TEST, str(dataset))
        images_dir = _prepare_images_dir(dataset=str(dataset), scene=str(scene), image_ids=image_ids, out_dir=out_dir, meta_map=meta_map)

        db_path = out_dir / "colmap_lightglue.db"
        sparse_root = out_dir / "sparse"
        sparse_root.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable,
            "scripts/lightglue_to_colmap.py",
            "--images-dir",
            str(images_dir),
            "--db-path",
            str(db_path),
            "--cache-dir",
            str(LIGHTGLUE_CACHE),
            "--max-keypoints",
            str(int(LIGHTGLUE_MAX_KPTS)),
            "--pairing",
            str(LIGHTGLUE_PAIRING),
            "--topk",
            str(int(LIGHTGLUE_TOPK)),
            "--min-inliers",
            str(int(LIGHTGLUE_MIN_INLIERS)),
        ]
        if OVERWRITE_SFM:
            cmd += ["--overwrite"]
        run(cmd, env=env)

        mapper_cmd = [
            "colmap",
            "mapper",
            "--database_path",
            str(db_path),
            "--image_path",
            str(images_dir),
            "--output_path",
            str(sparse_root),
            "--Mapper.ba_refine_principal_point",
            "0",
        ]
        print("+", shlex.join(mapper_cmd), flush=True)
        subprocess.run(mapper_cmd, check=True, cwd=str(REPO_ROOT))

if RUN_SFM_MATCHERS:
    run_recon_matcher(matcher="colmap", out_root=SFM_OUT_ROOT / "recon_colmap")
    run_recon_matcher(matcher="dino", out_root=SFM_OUT_ROOT / "recon_dino")
    run_recon_matcher(matcher="diffusion", out_root=SFM_OUT_ROOT / "recon_diffusion")
else:
    print("[skip] COLMAP/DINO/Diffusion SfM (RUN_SFM_MATCHERS=False)")

if RUN_LIGHTGLUE:
    run_lightglue_sfm(clusters_csv=CLUSTERS_FOR_SFM, out_root=SFM_OUT_ROOT / "recon_lightglue")
else:
    print("[skip] LightGlue SfM (RUN_LIGHTGLUE=False)")


+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/colmap_dense_clusters.py --clusters-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/results/clusters_cfg4_everything_selected.csv --cache-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/cache_retrieval_finetuned_test --data-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split_holdout_ets_sacrecoeur_stpeters --output-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_colmap --runner auto --matcher colmap --sparse-only
[info] Docker COLMAP does not support --Mapper.local_ba_min_tri_angle; disabling it.
Runner: docker
Clusters: 8
[cluster] 1/8: ETs/cluster_0001
[run] ETs/cluster_0001: images=10 matcher=colmap
+ docker run --rm --g

I20251230 14:56:50.396721    27 misc.cc:44] 
Feature extraction
I20251230 14:56:50.399441    38 sift.cc:720] Creating SIFT GPU feature extractor
I20251230 14:56:52.046226    39 feature_extraction.cc:260] Processed file [1/10]
I20251230 14:56:52.046317    39 feature_extraction.cc:263]   Name:            et_et000.png
I20251230 14:56:52.046326    39 feature_extraction.cc:272]   Dimensions:      480 x 640
I20251230 14:56:52.046331    39 feature_extraction.cc:275]   Camera:          #1 - SIMPLE_RADIAL
I20251230 14:56:52.046336    39 feature_extraction.cc:278]   Focal Length:    768.00px
I20251230 14:56:52.046350    39 feature_extraction.cc:282]   Features:        2319 (SIFT)
I20251230 14:56:52.071269    39 feature_extraction.cc:260] Processed file [2/10]
I20251230 14:56:52.071337    39 feature_extraction.cc:263]   Name:            et_et001.png
I20251230 14:56:52.071345    39 feature_extraction.cc:272]   Dimensions:      480 x 640
I20251230 14:56:52.071350    39 feature_extraction.cc:275]   

+ docker run --rm --gpus all -v /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025:/workspace -w /workspace colmap/colmap:latest colmap exhaustive_matcher --database_path outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_colmap/ETs_cluster_0001/colmap.db

== CUDA ==

CUDA Version 12.9.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.



I20251230 14:57:05.148552    27 misc.cc:44] 
Feature matching & geometric verification
I20251230 14:57:05.257699    28 feature_matching_utils.cc:78] Bind FeatureMatcherWorker to GPU device 0
I20251230 14:57:05.257848    28 sift.cc:1446] Creating SIFT GPU feature matcher
I20251230 14:57:05.305814    27 pairing.cc:180] Generating exhaustive image pairs...
I20251230 14:57:05.305876    27 pairing.cc:213] Processing block [1/1, 1/1]
I20251230 14:57:06.473440    27 feature_matching.cc:117] in 1.168s
I20251230 14:57:06.473507    27 timer.cc:90] Elapsed time: 0.022 [minutes]


+ docker run --rm --gpus all -v /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025:/workspace -w /workspace colmap/colmap:latest colmap mapper --database_path outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_colmap/ETs_cluster_0001/colmap.db --image_path outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_colmap/ETs_cluster_0001/images --output_path outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_colmap/ETs_cluster_0001/sparse --Mapper.ba_refine_principal_point 0 --Mapper.min_num_matches 15 --Mapper.min_model_size 3 --Mapper.init_min_num_inliers 15 --Mapper.init_min_tri_angle 2.0 --Mapper.tri_min_angle 1.5 --Mapper.filter_min_tri_angle 1.5 --Mapper.tri_ignore_two_view_tracks 1

== CUDA ==

CUDA Version 12.9.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Conta

...

[ok] pt_sacrecoeur_trevi_tajmahal/cluster_0002: sparse model ready at /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_dino/pt_sacrecoeur_trevi_tajmahal_cluster_0002/sparse/0
[cluster] 6/8: pt_sacrecoeur_trevi_tajmahal/cluster_0003
[run] pt_sacrecoeur_trevi_tajmahal/cluster_0003: images=74 matcher=dino
[dino] build db=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_dino/pt_sacrecoeur_trevi_tajmahal_cluster_0003/colmap_dino.db images=74 pairing=auto topk=10
[dino] cache_dir=cache/dino_features
[dino] model=facebook/dinov2-small layer=last max_side=512 fp16=True
[dino] matcher: keypoints=patch max_keypoints=2048 min_similarity=0.75 min_inliers=15
[warn] --single-camera requested but image sizes differ [(501, 501), (503, 375), (518, 381), (603, 809), (651, 989), (666, 999), (676, 1014), (6

I20251230 15:17:55.312337     1 incremental_pipeline.cc:264] Loading database
I20251230 15:17:55.599505     1 database_cache.cc:67] Loading rigs...
I20251230 15:17:55.653944     1 database_cache.cc:77]  0 in 0.055s
I20251230 15:17:55.654031     1 database_cache.cc:85] Loading cameras...
I20251230 15:17:55.803743     1 database_cache.cc:103]  74 in 0.150s
I20251230 15:17:55.803803     1 database_cache.cc:111] Loading frames...
I20251230 15:17:55.803818     1 database_cache.cc:126]  0 in 0.000s
I20251230 15:17:55.803822     1 database_cache.cc:134] Loading matches...
I20251230 15:17:55.805143     1 database_cache.cc:139]  514 in 0.001s
I20251230 15:17:55.805200     1 database_cache.cc:155] Loading images...
I20251230 15:17:55.860088     1 database_cache.cc:239]  74 in 0.055s (connected 74)
I20251230 15:17:55.860206     1 database_cache.cc:250] Building correspondence graph...
I20251230 15:17:55.863490     1 database_cache.cc:277]  in 0.003s (ignored 0)
I20251230 15:17:55.863919     1 tim

[ok] pt_sacrecoeur_trevi_tajmahal/cluster_0003: sparse model ready at /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_dino/pt_sacrecoeur_trevi_tajmahal_cluster_0003/sparse/1
[cluster] 7/8: pt_stpeters_stpauls/cluster_0001
[run] pt_stpeters_stpauls/cluster_0001: images=101 matcher=dino
[dino] build db=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ablation_holdout_ets_sacrecoeur_stpeters/sfm_matcher_compare/recon_dino/pt_stpeters_stpauls_cluster_0001/colmap_dino.db images=101 pairing=auto topk=10
[dino] cache_dir=cache/dino_features
[dino] model=facebook/dinov2-small layer=last max_side=512 fp16=True
[dino] matcher: keypoints=patch max_keypoints=2048 min_similarity=0.75 min_inliers=15
[warn] --single-camera requested but image sizes differ [(462, 621), (489, 489), (490, 490), (491, 491), (502, 375), (511, 379), (586, 786), (622, 462), (672, 1004), (67

Traceback (most recent call last):
  File "/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/scripts/colmap_dense_clusters.py", line 16, in <module>


KeyboardInterrupt: 